In [23]:
import anthropic
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PayloadSchemaType, PointStruct, SparseVectorParams, Document, Prefetch, FusionQuery
from qdrant_client import models
import pandas as pd
import cohere
from dotenv import load_dotenv
import os
import voyageai

In [24]:
load_dotenv()
qdrant_client = QdrantClient(url="http://localhost:6333")
VOYAGE_API_KEY = os.environ.get("VOYAGE_API_KEY")
voyageai_client = voyageai.Client(api_key=VOYAGE_API_KEY)

In [25]:
def get_embedding(text, model='voyage-3', input_type="document"):
        result = voyageai_client.embed(
                [text],
                model=model,
                input_type=input_type
        )
        return result.embeddings[0]

In [26]:
def retrieve_data(query, qdrant_client, k=5):
        query_embedding = get_embedding(query)
        results = qdrant_client.query_points(
                collection_name="Amazon-items-collection-01-hybrid-search",
                prefetch=[
                        Prefetch(
                                query=query_embedding,
                                using='voyage-3',
                                limit=10,
                        ),
                        Prefetch(
                                query=Document(
                                        text=query,
                                        model="qdrant/bm25"
                                ),
                                using='bm25',
                                limit=10,
                        ),
                ],
                query=FusionQuery(fusion='rrf'), #reciprocal rank fusion
                limit=k,
        )

        retrieved_context_ids = []
        retrieved_context = []
        similarity_scores = []
        retrieved_context_ratings = []

        for item in results.points:
                retrieved_context_ids.append(item.payload['parent_asin'])
                retrieved_context.append(item.payload['description'])
                retrieved_context_ratings.append(item.payload['average_rating'])
                similarity_scores.append(item.score)
        
        return {
                "retrieved_context_ids": retrieved_context_ids,
                "retrieved_context": retrieved_context,
                "retrieved_context_ratings": retrieved_context_ratings,
                "similarity_scores": similarity_scores,
        }

In [47]:
query = "can I get a laptop?"
results = retrieve_data(query, qdrant_client, k=20)

In [48]:
results #poor results

{'retrieved_context_ids': ['B0B87XZT6M',
  'B0B3MTQHD4',
  'B09Z6F54Y3',
  'B0BK91PLJG',
  'B09QYKBN5W',
  'B08JCX7MB7',
  'B0BLMWXBDY',
  'B0B9YQ6W6V',
  'B08FFFKBF3',
  'B09ZJ9LKCN',
  'B0CGCXG1HC',
  'B09YPTBTT5',
  'B09WYTX4LD',
  'B09P85LLHB',
  'B0B4WNFTVZ',
  'B09Q3LJMFT',
  'B0C9QD39WB',
  'B0BDK85Z37',
  'B09SQ6QKG4',
  'B0BZ4Q6Z21'],
 'retrieved_context': ['Adjustable Laptop Stand with 360° Rotating Base, ZKSIND Ergonomic Laptop Riser for Collaborative Work, Dual Rotary Shaft Fully Foldable for Easy Storage, Fits MacBook/All Laptops up to 16 inches ',
  "AWANFI Portable Laptop Charger with AC Outlet, 97Wh/100W Laptop Power Bank 27000mAh External Travel Battery Pack with LED Flashlight for Tablet,MacBook Pro, Notebooks, Smartphone 【Compact & Big Capacity】The 27000mAh 97Wh capacity portable power bank is equipped with 110V/100W AC output that can provide enough power to charge your laptop at full speed,or power up various devices such as phones, tablets, PSP etc. Portable size 

### Cohere reranking to improve the output

In [49]:
cohere_client = cohere.ClientV2()

In [50]:
to_rerank = results["retrieved_context"]
response = cohere_client.rerank(
        model="rerank-v4.0-pro",
        query=query,
        documents=to_rerank,
        top_n=20
)
response #ordered reranked item

V2RerankResponse(id='5a41e2bc-adf1-465a-98e7-a1f320b913df', results=[V2RerankResponseResultsItem(index=16, relevance_score=0.8575773), V2RerankResponseResultsItem(index=8, relevance_score=0.81440735), V2RerankResponseResultsItem(index=7, relevance_score=0.7531763), V2RerankResponseResultsItem(index=13, relevance_score=0.726115), V2RerankResponseResultsItem(index=1, relevance_score=0.71827835), V2RerankResponseResultsItem(index=15, relevance_score=0.7166947), V2RerankResponseResultsItem(index=12, relevance_score=0.71191186), V2RerankResponseResultsItem(index=18, relevance_score=0.70383453), V2RerankResponseResultsItem(index=3, relevance_score=0.6856128), V2RerankResponseResultsItem(index=4, relevance_score=0.67198735), V2RerankResponseResultsItem(index=11, relevance_score=0.67198735), V2RerankResponseResultsItem(index=0, relevance_score=0.66332006), V2RerankResponseResultsItem(index=6, relevance_score=0.6184117), V2RerankResponseResultsItem(index=9, relevance_score=0.6110107), V2RerankR

In [51]:
reranked_results = [to_rerank[item.index] for item in response.results]
for item in reranked_results:
        print(item) #better result

Lenovo 2023 Newest IdeaPad 3i Laptop, 14" FHD IPS Display, Intel Core i5-1235U(up to 4.40GHz,10 Cores, 12 Threads), 8GB RAM, 256GB SSD, Intel Iris Xe Graphics, Fingerprint, Wi-Fi 6, Windows 11 Home 【Processor】Equipped with Intel Core i5-1235U (1.30 GHz, up to 4.40GHz Max Boost, 10 Cores, 12 Threads, 12 MB Cache) Processor. The high-performance configuration allows you enjoy impressive creating, which is an ideal home office laptop to get things done fast with high performance, instant responsiveness and best-in-class connectivity. 【14" HD Display】The 14" FHD (1920x1080) IPS 300nits Anti-glare display with 4-sided narrow bezels lets you see more and do more with better color accuracy and contrast. 【Upgrade】8GB RAM is designed for basic tasks, the high-bandwith DDR4 RAM run your applications smoothly, as well as multiple programs and files all at once. 256GB storage capacity is suitable for saving all your files and provides enough space to save more data. 【Connectivity】Wireless/Wired co